## 1. Importação de Bibliotecas

In [ ]:
import warnings
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

warnings.filterwarnings("ignore")

## 2. Configuração do Ambiente

In [ ]:
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

BASE_DIR = Path(".")

FIGURES_DIR = BASE_DIR / "figures"
TABLES_DIR = BASE_DIR / "tables"

FIGURES_DIR.mkdir(exist_ok=True)
TABLES_DIR.mkdir(exist_ok=True)

## 3. Funções Auxiliares

In [ ]:
def load_excel(filename: str) -> pd.DataFrame:
    path = BASE_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")
    return pd.read_excel(path)


def save_figure(fig: plt.Figure, filename: str) -> None:
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")

## 4. Carregamento dos Dados

In [ ]:
df_ativos = load_excel("Estudantes_ativos_EP_2025.xlsx")
df_inativos = load_excel("Estudantes_inativos_EP_2025.xlsx")
df_concat = load_excel("Estudantes_EP_2025_concat.xlsx")

df_inativos = df_inativos[df_inativos['Ano_Ingresso'] >= 2012]
df_concat = df_concat[df_concat['Ano_Ingresso'] >= 2012]

## 5. IRA por Coorte (Estudantes Ativos)

In [ ]:
if {"Ano_Ingresso", "IRA"}.issubset(df_ativos.columns):
    df_ativos["Ano_Ingresso"] = pd.to_numeric(df_ativos["Ano_Ingresso"], errors="coerce")
    df_valid = df_ativos.dropna(subset=["Ano_Ingresso", "IRA"])
    
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.boxplot(data=df_valid, x="Ano_Ingresso", y="IRA", ax=ax)
    ax.set_xlabel("Ano de ingresso")
    ax.set_ylabel("IRA")

    save_figure(fig, "fig_8_boxplot_ira_por_coorte_ativos.png")
    plt.show()

## 6. Média de Porcentagem Concluída por Coorte

In [ ]:
if {"Ano_Ingresso", "IRA", "Porcentagem_Concluido_SIGA"}.issubset(df_ativos.columns):
    df_ativos["Ano_Ingresso"] = pd.to_numeric(df_ativos["Ano_Ingresso"], errors="coerce")
    df_valid = df_ativos.dropna(subset=["Ano_Ingresso", "IRA"])
    
    if "Porcentagem_Concluido_SIGA" in df_valid.columns:
        media_concluido = (
            df_valid.groupby("Ano_Ingresso")["Porcentagem_Concluido_SIGA"]
            .mean()
            .reset_index()
        )
        
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(
            media_concluido["Ano_Ingresso"],
            media_concluido["Porcentagem_Concluido_SIGA"],
            marker="o"
        )
        ax.set_xlabel("Ano de ingresso")
        ax.set_ylabel("Média de % concluído")
        
        save_figure(fig, "fig_9_media_concluido_por_coorte.png")
        plt.show()

## 7. IRA por Sexo

In [ ]:
if {"IRA", "Sexo"}.issubset(df_concat.columns):

    df_box = df_concat.dropna(subset=["IRA", "Sexo"])

    medias = df_box.groupby("Sexo")["IRA"].mean()
    mean_M = medias.get("M", float("nan"))
    mean_F = medias.get("F", float("nan"))

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.boxplot(data=df_box, x="Sexo", y="IRA", ax=ax)

    ax.text(0, mean_M + 300, 
            f"Média: {mean_M:.1f}", 
            ha="center", color="#1a1a1a", fontsize=10, fontweight="bold")

    ax.text(1, mean_F + 300, 
            f"Média: {mean_F:.1f}", 
            ha="center", color="#1a1a1a", fontsize=10, fontweight="bold")

    ax.set_xlabel("Sexo")
    ax.set_ylabel("IRA")

    save_figure(fig, "fig_10_boxplot_ira_media_por_sexo.png")
    plt.show()

## 8. Criação de Variáveis de Situação e Evasão

In [ ]:
def adicionar_situacao_e_evadido_flag(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "Situacao" not in df.columns:
        def class_situacao(row):
            status = str(row.get("Status", "")).strip()
            if status in ["Cursando", "Candidato à Formatura"]:
                return "Ativo"
            elif pd.isna(row.get("Ano_Egresso")):
                return "Ativo"
            else:
                return "Inativo"
        df["Situacao"] = df.apply(class_situacao, axis=1)
    if "Evadido_flag" not in df.columns:
        df["Evadido_flag"] = (df["Situacao"] == "Inativo").astype(int)
    return df

df_concat = adicionar_situacao_e_evadido_flag(df_concat)

## 9. IRA vs Porcentagem Concluída por Situação

In [ ]:
cols_needed = {"IRA", "Porcentagem_Concluido_SIGA", "Situacao"}
if cols_needed.issubset(df_concat.columns):
    df_scatter = df_concat.dropna(subset=list(cols_needed))
    df_scatter = df_scatter.loc[df_scatter['Ano_Ingresso'] >= 2012]
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.scatterplot(
        data=df_scatter,
        x="Porcentagem_Concluido_SIGA",
        y="IRA",
        hue="Situacao",
        alpha=0.7,
        ax=ax,
    )
    ax.set_xlabel("Porcentagem concluída no curso (%)")
    ax.set_ylabel("IRA")
    ax.legend(title="Situação", bbox_to_anchor=(1.05, 1), loc="upper left")

    save_figure(fig, "fig_12_scatter_ira_concluido_por_situacao.png")
    plt.show()

## 10. Matriz de Correlação - Desempenho e Progresso

In [ ]:
cols_corr = [
    "IRA",
    "Porcentagem_Concluido_SIGA",
    "Porcentagem_Inscrito",
    "Porcentagem_Aprovado",
    "Porcentagem_Reprovado",
    "Numero_Horas_Inscritas_Resultado",
    "Horas_Aprovadas",
]

cols_corr_presentes = [c for c in cols_corr if c in df_concat.columns]

if len(cols_corr_presentes) >= 2:
    sub = df_concat[cols_corr_presentes].apply(pd.to_numeric, errors="coerce").dropna()
    corr = sub.corr()

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)

    save_figure(fig, "fig_13_matriz_correlacao_desempenho_progresso.png")
    plt.show()

## 11. Preparação de Rótulos dos Status de Evasão

In [ ]:
mapa_renome = {
    "Cancelado": "Cancelado",
    "Cancelado Reaproveitamento Vaga Vestibular": "Cancelado – Reap. Vestibular",
    "Em Recurso": "Em Recurso",
    "Falecido": "Falecido",
    "Inelegível por Análise Socioeconômica": "Inelegível – Socioeconômico",
    "Perda de Vaga Desempenho Mínimo": "Perda – Desempenho",
    "Perda de Vaga Não Confirmação Matricula": "Perda – Não Conf. Matrícula",
    "Perda de Vaga Rematrícula": "Perda – Rematrícula",
    "Transferência Externa": "Transferência Externa",
    "Transferência Interna": "Transferência Interna",
}

df_inativos["Status_Abrev"] = df_inativos["Status"].replace(mapa_renome)

## 12. Motivos de Evasão (Percentual)

In [ ]:
col_motivo = "Status_Abrev"

if col_motivo in df_inativos.columns:
    dist_motivo = (
        df_inativos[col_motivo]
        .value_counts(normalize=True)
        .mul(100)
        .sort_values(ascending=True)
        .reset_index()
    )
    dist_motivo.columns = ["Motivo", "Proporcao"]
    
    fig, ax = plt.subplots(figsize=(10, max(5, 0.4 * len(dist_motivo))))
    sns.barplot(data=dist_motivo, x="Proporcao", y="Motivo", ax=ax)
    ax.set_xlabel("Proporção (%)")
    ax.set_ylabel("Motivo de evasão")
    
    save_figure(fig, "fig_14_motivos_evasao_percentual.png")
    plt.show()

## 13. Tempo até Evasão (Percentual)

In [ ]:
if "Tempo_Evasao" in df_inativos.columns:
    tempo_dist = (
        df_inativos["Tempo_Evasao"]
        .value_counts(normalize=True)
        .mul(100)
        .sort_index()
        .reset_index()
    )
    tempo_dist.columns = ["Tempo", "Proporcao"]
    
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(tempo_dist["Tempo"], tempo_dist["Proporcao"])
    ax.set_xlabel("Anos até evasão")
    ax.set_ylabel("Proporção (%)")
    
    save_figure(fig, "fig_15_tempo_ate_evasao_percentual.png")
    plt.show()

## 14. Tempo até Evasão por Status

In [ ]:
if {"Tempo_Evasao", "Status_Abrev"}.issubset(df_inativos.columns):
    df_tempo_status = df_inativos.dropna(subset=["Tempo_Evasao", "Status_Abrev"])
    
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(data=df_tempo_status, x="Status_Abrev", y="Tempo_Evasao", ax=ax)
    ax.set_xlabel("Status")
    ax.set_ylabel("Anos até evasão")
    plt.xticks(rotation=45, ha="right")
    
    save_figure(fig, "fig_16_tempo_evasao_por_status.png")
    plt.show()

## 15. IRA dos Evadidos por Status

In [ ]:
if {"IRA", "Status_Abrev"}.issubset(df_inativos.columns):
    df_ira_status = df_inativos.dropna(subset=["IRA", "Status_Abrev"])
    
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(data=df_ira_status, x="Status_Abrev", y="IRA", ax=ax)
    ax.set_xlabel("Status")
    ax.set_ylabel("IRA")
    plt.xticks(rotation=45, ha="right")
    
    save_figure(fig, "fig_17_ira_evadidos_por_status.png")
    plt.show()